# PCA + k-NN on MNIST

Using Principal Component Analysis to compress MNIST's 784-pixel images down to a handful of components, then classifying digits with k-NN on the reduced representation — and checking the result against a random-guess baseline.

## Preparation Steps

In [1]:
# Import all necessary python packages
import numpy as np
#import os
#import pandas as pd
#import matplotlib.pyplot as plt
#from matplotlib.colors import ListedColormap
#from sklearn.linear_model import LogisticRegression

In [2]:
# we load the data set directly from scikit learn
#
# note: this operation may take a few seconds. If for any reason it fails we
# can revert back to loading from local storage.

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split


X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
y = y.astype(int)
X = ((X / 255.) - .5) * 2    # Line A for Question Q1-5
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=10000, random_state=123, stratify=y)


## <font color = 'blue'> Section 1. Inspecting the Dataset </font>

This section inspects the MNIST dataset itself: the size of the training and test sets, the number of attributes (pixels) per image, the number of distinct digit labels, how the attribute count relates to the images' pixel dimensions, and what the data-loading code is doing under the hood.

In [3]:
# Question 1-1
print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

# Question 1-2
print("Number of attributes:", X.shape[1])

# The .shape() method of Numpy arrays returns (rows, columns). The rows are the data points, and the columns are the attributes.
# .shape[0] returns the number of rows (number of data points), while .shape[1] returns the number of columns (number of attributes)

# Question 1-3
print("Unique labels:", np.unique(y_train))
print("Number of unique labels:", len(np.unique(y_train)))

# Question 1-4
# The number of attributes is 784 as found in Question 1-2. The size of the images in the MNIST is 28 x 28 pixels. 28 x 28 = 784.
# The number of pixels in each MNIST images is equal to the number of attributes. Each attribute contains the intensity value of a single pixel.
# When the image is loaded, it is flattened into a 1-dimensional vector of 784 attributes.

# Question 1-5
# Line A serves to normalize the attribute values around 0. Each attribute (pixel) value from the MNIST dataset ranges from 0 to 255.
# Line A divides the attribute (pixel) value by 255, this will make the attribute values range from 0 to 1. We then subtract the result
# by 0.5. This will move the range from -0.5 to 0.5. Then by multiplying the attribute value by 2, we change the range of its values
# to -1 to 1. The reason we do this is because machine learning models (like logistic regression) converges faster and more stably when
# the features are scaled around 0. Also, by scaling the features around 0, it prevents larger attribute values from dominating the
# learning process. In the context of PCA, PCA finds the directions of maximum variance in the dataset. It does this by calculating the
# covariance matrix of the features. The scale of the features directly affects the variance and therefore the principal components it
# finds. The features with larger values dominate the variance, and the PCA becomes biased towards them. By normalizing them around 0,
# the principal components found will be based on variance, not feature magnitude.

Training set size: 60000
Test set size: 10000
Number of attributes: 784
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Number of unique labels: 10


## <font color = 'blue'> Section 2. PCA on MNIST </font>

MNIST's 784 raw pixel attributes make k-NN impractical due to the curse of dimensionality, so this section uses scikit-learn's PCA to compress the images down to a small number of transformed components ($l$, here chosen around 10 — the number of digit classes) before classification.

In [4]:
# Question 2
# Import PCA
from sklearn.decomposition import PCA
# Initialize l attributes
l = 10
# Intialize PCA object with n_components set to variable l
pca = PCA(n_components=l)
# Transform X_train and X_test using PCA.fit_transform to compress number of attributes down to l
X_train_transformed = pca.fit_transform(X_train)
X_test_transformed = pca.transform(X_test)
# Output Original # of Attributes vs After Being Transformed
print("Original shape (train):", X_train.shape)
print("Transformed shape (train):", X_train_transformed.shape)
print("Original shape (test):", X_test.shape)
print("Transformed shape (test):", X_test_transformed.shape)

Original shape (train): (60000, 784)
Transformed shape (train): (60000, 10)
Original shape (test): (10000, 784)
Transformed shape (test): (10000, 10)


## <font color = 'blue'> Section 3. kNN on MNIST attributes from PCA </font>

With the PCA-transformed dataset in hand, this section fits a k-NN classifier on it, evaluates its classification accuracy on the test set, compares that result against the accuracy expected from random guessing, and experiments with different values of $k$ and other k-NN hyperparameters to see how they affect performance.

In [5]:
# Question 3-1
# Import KNN Classifier Class
from sklearn.neighbors import KNeighborsClassifier
# Initialize k - hyperparameter
k = 5
# Initialize k-NN Classifier - Neighbors = k, p=2 & metric='minkowski' - set the distance measure to Euclidean measure
knn = KNeighborsClassifier(n_neighbors=k, p=2, metric='minkowski')
# Fit k-NN Classifier to Training Data Sets: Transformed X and Y
knn.fit(X_train_transformed, y_train)

# Question 3-2
# Import Accuracy Score Class
from sklearn.metrics import accuracy_score
# Predict Y labels based on Transformed Test X dataset
y_pred = knn.predict(X_test_transformed)
# Use accuracy_score method on Test Y and Predicted Y
acc = accuracy_score(y_test, y_pred)
# Output K Value & Accuracy Score
print(f"k = {k} - Test Accuracy: {acc:.4f}")

# Question 3-3
# There are 10 unique labels in the MNIST dataset as determined by Question 1-3. If we were to randomly assign labels to the test points.
# We would be accurate about 10% (1 / 10 = 10%) of the time. Now this expected accuracy of about 10% is in the long run, the accuracy of doing this in the
# short run can vary. Yes, the accuracy of the kNN classifier used in Question 3-2 is about 93% which is significantly higher than the
# expected accuracy of randomly assigning labels (10%).

k = 5 - Test Accuracy: 0.9346


In [7]:
# Question 3-4
# Import Grid Search CV Class
from sklearn.model_selection import GridSearchCV
# Initialize Grid of Parameters to Randomly Sample From
param_grid = {
    "n_neighbors": [1, 3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
    "leaf_size": [30, 50]
}
# Initialize k-NN Classifier without passing any parameters
kNN = KNeighborsClassifier()
# Initialize an Instance Grid Search CV Object
grid = GridSearchCV(
    kNN, param_grid,
    cv=3, n_jobs=-1, verbose=0, scoring="accuracy"
)
# Using Grid Search CV Object Fit to Transformed X Training Dataset and Y Training Dataset
grid.fit(X_train_transformed, y_train)
# Output Findings
print("Best Parameters: ", grid.best_params_)
print(f"Best Accuracy: {grid.best_score_:.4f}")

# Get Best Model Settings
best_kNN = grid.best_estimator_
# Predict Y Labels - Using Transformed X Test Dataset
y_pred = best_kNN.predict(X_test_transformed)
# Calculate Accuracy Score for Best kNN Model on Test Data
best_kNN_acc = accuracy_score(y_test, y_pred)
# Output Findings
print(f"Best kNN Model - Test Accuracy: {best_kNN_acc:.4f}")

Best Parameters:  {'leaf_size': 30, 'metric': 'euclidean', 'n_neighbors': 7, 'weights': 'distance'}
Best Accuracy: 0.9297
Best kNN Model - Test Accuracy: 0.9351
